# FIFA 2026 World Cup — Prediction System
## Notebook 4: XGBoost Poisson Model + Ensemble

### Where we are
Notebook 3 built a Poisson GLM baseline — two linear models predicting
expected goals (λ) for home and away teams. The GLM is interpretable and
well calibrated but assumes a linear relationship between features and log(λ).

### What this notebook adds

**XGBoost Poisson**
Same targets (home_score, away_score), same features, but gradient-boosted
trees instead of a linear model. XGBoost captures non-linear interactions
the GLM misses — for example that Elo difference matters more at extremes,
or that squad value only kicks in above a certain threshold.

**Ensemble**
We blend GLM and XGBoost λ estimates:
`λ_final = α · λ_GLM + (1-α) · λ_XGB`
α is tuned on a validation set to minimise prediction error.

**Calibration**
We evaluate both models and the ensemble using Ranked Probability Score (RPS)
— a metric that rewards not just getting the winner right but having the
right confidence level.

### Inputs
- `training_predictions_glm.csv` — df_model_v2 with GLM λ predictions
- `glm_home_v2.pkl` / `glm_away_v2.pkl` — fitted GLM objects
- `features_v2.pkl` — feature list

### Outputs
- Ensemble λ predictions for all training matches
- Calibration report — GLM vs XGBoost vs Ensemble
- Saved XGBoost models and ensemble weights for Notebook 5

In [1]:
# Load 
import pickle
import pandas as pd
from pathlib import Path

processed_dir = Path.home() / 'wc2026/data/processed'
models_dir    = Path.home() / 'wc2026/models'

# Reload df_model_v2
df_model_v2 = pd.read_csv(
    processed_dir / 'training_predictions_glm.csv', 
    parse_dates=['date']
)

# Reload models
with open(models_dir / 'glm_home_v2.pkl', 'rb') as f:
    glm_home_v2 = pickle.load(f)

with open(models_dir / 'glm_away_v2.pkl', 'rb') as f:
    glm_away_v2 = pickle.load(f)

with open(models_dir / 'features_v2.pkl', 'rb') as f:
    FEATURES_V2 = pickle.load(f)

In [3]:
print("Df_model_v2 Sahpe:", df_model_v2.shape)

Df_model_v2 Sahpe: (24715, 28)


In [4]:
# Confirms features and targets are present
print("Features:", FEATURES_V2)

print("\nMissing values:")
for f in FEATURES_V2:
    print(f" {f:<25} missing: {df_model_v2[f].isna().sum()}")

print("\nTargets:")
print(f"home_score missing: {df_model_v2['home_score'].isna().sum()}")
print(f"away_score missing: {df_model_v2['away_score'].isna().sum()}")

print("\nGLM predictions present:")
print(f" lambda_home_glm: {'Yes' if 'lambda_home_glm' in df_model_v2.columns else 'No'}")
print(f" lambda_away_glm: {'Yes' if 'lambda_away_glm' in df_model_v2.columns else 'No'}")
print(f"\nShape: {df_model_v2.shape}")

Features: ['elo_diff', 'squad_value_diff', 'home_form', 'away_form', 'home_advantage']

Missing values:
 elo_diff                  missing: 1
 squad_value_diff          missing: 1
 home_form                 missing: 1
 away_form                 missing: 1
 home_advantage            missing: 1

Targets:
home_score missing: 1
away_score missing: 1

GLM predictions present:
 lambda_home_glm: Yes
 lambda_away_glm: Yes

Shape: (24715, 28)


In [5]:
# Drop the single null row
before = len(df_model_v2)

df_model_v2 = df_model_v2.dropna(
    subset = FEATURES_V2 + ['home_score', 'away_score']
).reset_index(drop=True)

print(f"Before drop: {before:,}")
print(f"After drop: {len(df_model_v2):,}")
print(f"Dropped: {before - len(df_model_v2)}")

# Confirm clean
print("\nMissing values reamining:")
missing = df_model_v2[FEATURES_V2].isna().sum()
print(missing[missing>0].to_strings() if missing[missing>0].any() else 'None - all clean')

Before drop: 24,715
After drop: 24,714
Dropped: 1

Missing values reamining:
None - all clean


In [7]:
print(f"Date range: {df_model_v2.date.min(), '->', df_model_v2.date.max()}")

Date range: (Timestamp('1916-07-02 00:00:00'), '->', Timestamp('2026-06-10 00:00:00'))


## Train / Validation Split

We split by **date** not randomly.

Random split would leak future information into training — a match from 2024
could end up in training while a 2018 match is in validation, which makes
no sense temporally. The model should only ever learn from the past.

**Split point: January 2022**
- **Train** — everything before 2022
- **Validation** — 2022 onwards, including the 2022 World Cup

This gives us ~2 years of holdout data with the most relevant recent
matches to validate against — including a full World Cup cycle.

In [8]:
#  Split: train on everything before 2022, validate on 2022 onwards.

train = df_model_v2[df_model_v2['date']<'2022-01-01'].copy()
val = df_model_v2[df_model_v2['date']>='2022-01-01'].copy()


print(f"Train: {len(train):,} matches  ({train['date'].min().date()} -> {train['date'].max().date()})")
print(f"Val:   {len(val):,} matches  ({val['date'].min().date()} -> {val['date'].max().date()})")
print(f"\nTrain home score mean: {train['home_score'].mean():.3f}")
print(f"Val   home score mean: {val['home_score'].mean():.3f}")
print(f"\nTrain away score mean: {train['away_score'].mean():.3f}")
print(f"Val   away score mean: {val['away_score'].mean():.3f}")


Train: 20,258 matches  (1916-07-02 -> 2021-12-31)
Val:   4,456 matches  (2022-01-01 -> 2026-06-10)

Train home score mean: 1.691
Val   home score mean: 1.601

Train away score mean: 1.116
Val   away score mean: 1.114


## XGBoost Poisson Model

### Why XGBoost after the GLM?
The GLM assumes a **linear** relationship between features and log(λ).
XGBoost relaxes this — gradient boosted trees can find non-linear patterns:
- Elo difference matters more at the extremes than in the middle
- Squad value only kicks in above a certain threshold
- Form interacts differently depending on the strength gap between teams

Same targets (`home_score`, `away_score`), same features, but a fundamentally
different learning algorithm.

### Native XGBoost API vs Sklearn interface
We use the **native XGBoost API** (`xgb.train`) rather than the sklearn
interface (`XGBRegressor.fit`). The native API supports early stopping
cleanly and gives full control over training.

| Sklearn interface | Native API |
|---|---|
| `XGBRegressor(**params)` | `params = {...}` |
| `.fit(X, y)` | `xgb.train(params, dtrain)` |
| `n_estimators` in params | `num_boost_round` in `train()` |
| `.predict(X)` | `.predict(DMatrix(X))` |

### DMatrix
XGBoost's own internal data format. Bundles features, labels, and sample
weights into one optimised object. Required by the native API.
```python
dtrain = xgb.DMatrix(X_train, label=y_train, weight=w_train)
```

### Key parameters
| Parameter | Value | Reason |
|---|---|---|
| `objective` | `count:poisson` | Treats goals as count data, same assumption as GLM |
| `max_depth` | 4 | Shallow trees — avoids overfitting on small feature set |
| `learning_rate` | 0.05 | Small steps — more trees but better generalisation |
| `num_boost_round` | 1000 | Upper limit — early stopping finds true optimum |
| `early_stopping_rounds` | 50 | Stop if no val improvement for 50 consecutive rounds |
| `subsample` | 0.8 | Row sampling per tree — regularisation |
| `colsample_bytree` | 0.8 | Feature sampling per tree — regularisation |

### Two separate models
Same design as the GLM — one model for `home_score`, one for `away_score`.
Captures the asymmetry between scoring and conceding independently.

### Sample weights
`match_weight` passed as `weight` in both train and validation DMatrix —
World Cup matches (1.0) influence the trees more than friendlies (0.3),
consistent with the GLM approach.

In [11]:
# XGBoost Poisson
# We fit two separate models — home and away — same as the GLM.
import xgboost as xgb

X_train = train[FEATURES_V2]
X_val = val[FEATURES_V2]

y_train_home = train['home_score']
y_train_away = train['away_score']
y_val_home = val['home_score']
y_val_away = val['away_score']

w_train = train['match_weight']
w_val = val['match_weight']

# Build DMatrix objects 
# DMatrix bundles features, labels, and weights into one optimised object
# Required by the native XGBoost API

dtrain_home = xgb.DMatrix(X_train, label = y_train_home, weight = w_train)
dval_home = xgb.DMatrix(X_val, label = y_val_home, weight = w_val)

dtrain_away = xgb.DMatrix(X_train, label = y_train_away, weight = w_train)
dval_away = xgb.DMatrix(X_val, label = y_val_away, weight = w_val)

# XGBoost parameters
params = {
    'objective':            'count:poisson',
    'max_depth':             4,
    'learning_rate':         0.05,
    'subsample':             0.8,
    'colsample_bytree':      0.8,
    'seed':                  42,
    'verbosity':             0, 
}
# Home model
xgb_home = xgb.train(
    params,
    dtrain_home,
    num_boost_round = 1000,
    evals = [(dval_home, 'val')],
    early_stopping_rounds = 50,
    verbose_eval = False
)

# Away model
xgb_away = xgb.train(
    params,
    dtrain_away,
    num_boost_round = 1000,
    evals = [(dval_away, 'val')],
    early_stopping_rounds = 50,
    verbose_eval = False
)

print("XGBoost home model fitted")
print(f" Best iteration home: {xgb_home.best_iteration}")
print(f" Best val score: {xgb_home.best_score:.4f}")

print("XGBoost away model fitted")
print(f" Best iteration away: {xgb_away.best_iteration}")
print(f" Best val score: {xgb_away.best_score:.4f}")


XGBoost home model fitted
 Best iteration home: 460
 Best val score: 1.5231
XGBoost away model fitted
 Best iteration away: 169
 Best val score: 1.3144


## Generating XGBoost λ Predictions

### What we're doing
The fitted XGBoost models output **λ (expected goals)** for every match —
same as the GLM did in Notebook 3. We generate predictions on both the
training set and validation set so we can compare XGBoost against the GLM.

### Why iteration_range?
Early stopping found the optimal number of trees:
- Home model: 460 trees
- Away model: 169 trees

Without `iteration_range`, XGBoost would predict using all 1000 trees —
including the ones that started overfitting. We tell it explicitly to
stop at the best iteration:
```python
iteration_range=(0, xgb_home.best_iteration)
```

### Why DMatrix for prediction?
The native API requires DMatrix at every step — not just training.
For prediction we only need features, no labels or weights:
```python
dpred = xgb.DMatrix(X)           # features only
model.predict(dpred, ...)         # predict λ
```

### What we're checking
Mean predicted λ should sit close to actual mean goals — same sanity
check we did for the GLM. If XGBoost is systematically over or under
predicting it signals a calibration problem before we even get to
the ensemble step.

In [12]:
# Generate XGBoost λ predictions 
# Predict on both train and val sets.
# Note: native API requires DMatrix for prediction.
# best_iteration tells XGBoost to use the optimal tree count,
# not all 1000 — avoids using an overfit model.

# Build prediction DMatrices (no labels needed for prediction)
dpred_train = xgb.DMatrix(train[FEATURES_V2])
dpred_val = xgb.DMatrix(val[FEATURES_V2])

# Generate predictions at best iteration
train['lambda_home_xgb'] = xgb_home.predict(dpred_train, iteration_range = (0, xgb_home.best_iteration))
train['lambda_away_xgb'] = xgb_away.predict(dpred_train, iteration_range = (0, xgb_away.best_iteration))

val['lambda_home_xgb'] = xgb_home.predict(dpred_val, iteration_range = (0, xgb_home.best_iteration))
val['lambda_away_xgb'] = xgb_away.predict(dpred_val, iteration_range = (0, xgb_away.best_iteration))

print("Validation set - mean λ comparison:")
print(f" Actual home goals: {val['home_score'].mean():.3f}")
print(f" GLM λ_home_mean: {val['lambda_home_glm'].mean():.3f}")
print(f" XGB λ_home_mean: {val['lambda_home_xgb'].mean():.3f}")

print(f" Actual away goals: {val['away_score'].mean():.3f}")
print(f" GLM λ_away_mean: {val['lambda_away_glm'].mean():.3f}")
print(f" XGB λ_away_mean: {val['lambda_away_xgb'].mean():.3f}")



Validation set - mean λ comparison:
 Actual home goals: 1.601
 GLM λ_home_mean: 1.704
 XGB λ_home_mean: 1.721
 Actual away goals: 1.114
 GLM λ_away_mean: 1.116
 XGB λ_away_mean: 1.124


## λ Prediction Comparison — GLM vs XGBoost

**Away goals — both models well calibrated**
Actual 1.114 vs GLM 1.116 and XGBoost 1.124 — essentially perfect.
Away scoring is consistent and both models capture it cleanly.

**Home goals — both models slightly overpredict**
Actual 1.601 vs GLM 1.704 and XGBoost 1.721.
Both models were trained on data with higher home scoring (1.691)
but the validation period (2022 onwards) shows lower home scoring (1.601).
This reflects the **post-COVID home advantage erosion** — empty stadiums
in 2020-21 changed scoring patterns and the effect has partially persisted.

**XGBoost slightly more aggressive than GLM**
XGBoost overshoots home goals more than the GLM (1.721 vs 1.704).
The GLM's linearity constraint acts as a natural brake against
chasing the training data too hard.

This is exactly why we ensemble — GLM conservatism balances
XGBoost aggressiveness. Calibration will correct the remaining bias.

### Feature Importance

In [15]:
# Feature importance 
# Feature importance tells us which features XGBoost relied on most
# when building its trees.
# We use 'gain' — the average improvement in loss brought by a feature
# across all trees it appears in. More meaningful than 'weight' (frequency)
# because it measures actual predictive contribution not just usage.

importance_home = pd.DataFrame({
    'feature': FEATURES_V2,
    'gain_home' : [xgb_home.get_score(importance_type='gain').get(f,0)
                  for f in FEATURES_V2]
}).sort_values('gain_home', ascending=False)

importance_away = pd.DataFrame({
    'feature': FEATURES_V2,
    'gain_away': [xgb_away.get_score(importance_type='gain').get(f,0)
                 for f in FEATURES_V2]
}).sort_values('gain_away', ascending=False)

print("Home model — feature importance (gain):")
print(importance_home.to_string(index=False))

print("\nAway model — feature importance (gain):")
print(importance_away.to_string(index=False))

Home model — feature importance (gain):
         feature  gain_home
        elo_diff  15.922933
squad_value_diff   6.031110
       away_form   4.699990
  home_advantage   3.064381
       home_form   2.862700

Away model — feature importance (gain):
         feature  gain_away
        elo_diff  24.960110
  home_advantage  10.323715
squad_value_diff   9.453220
       home_form   6.796612
       away_form   4.024403


## Ensemble — Blending GLM and XGBoost

### Why ensemble?
No single model is best at everything. The GLM and XGBoost make different
kinds of errors — blending them produces a more robust estimate than
either alone.

- **GLM** — conservative, linear, well calibrated on average
- **XGBoost** — aggressive, non-linear, better at extremes and thresholds
- **Ensemble** — balances both, reduces individual model weaknesses

### How we blend
```python
λ_final = α · λ_GLM + (1-α) · λ_XGB
```
A simple weighted average of the two λ estimates.
α = 1.0 means pure GLM, α = 0.0 means pure XGBoost.

### How we find the best α
We search α from 0.0 to 1.0 in steps of 0.05 and measure
**Mean Absolute Error** between predicted λ and actual goals
on the validation set. The α that minimises combined home
and away MAE is chosen.

This is tuned on unseen data (2022 onwards) so the blend
weight reflects real out-of-sample performance — not
how well the models fit their own training data.

In [17]:
# Ensemble — blend GLM and XGBoost λ
# We blend the two λ estimates:
# λ_final = α · λ_GLM + (1-α) · λ_XGB
# α is tuned on the validation set by minimising Mean Absolute Error
# between predicted λ and actual goals.
# We search across a range of α values from 0 (pure XGBoost)
# to 1 (pure GLM) in steps of 0.05.

import numpy as np

alphas = np.arange(0, 1.05, 0.05)

mae_home = []
mae_away = []

for alpha in alphas:
    # Blend λ on validation set
    lh_blend = alpha * val['lambda_home_glm'] + (1 - alpha) * val['lambda_home_xgb']
    la_blend = alpha * val['lambda_away_glm'] + (1 - alpha) * val['lambda_away_xgb']

    # MAE agains actual goals
    mae_home.append(np.mean(np.abs(lh_blend - val['home_score'])))
    mae_away.append(np.mean(np.abs(la_blend - val['away_score'])))

# Combined MAE - average of home and away
mae_combined = [( h + a )/ 2 for h, a in zip(mae_home, mae_away)]

# Best alpha 
best_idx = np.argmin(mae_combined)
best_alpha = alphas[best_idx]
best_mae = mae_combined[best_idx]

print("Alpha search results:")
print(f"{'Alpha': >8} {'MAE home':>10} {'MAE away':>10} {'MAE combined': >14}")
print("-" * 46)
for a, h, aw, c in zip(alphas, mae_home, mae_away, mae_combined):
    marker = " ← best" if a == best_alpha else ""
    print(f"{a:>8.2f} {h:>10.4f} {aw:>10.4f} {c:>14.4f}{marker}")

print(f"\nBest alpha: {best_alpha:.2f}")
print(f"  Pure GLM    (α=1.0): MAE = {mae_combined[-1]:.4f}")
print(f"  Pure XGBoost(α=0.0): MAE = {mae_combined[0]:.4f}")
print(f"  Ensemble    (α={best_alpha:.2f}): MAE = {best_mae:.4f}")

Alpha search results:
   Alpha   MAE home   MAE away   MAE combined
----------------------------------------------
    0.00     1.0418     0.8388         0.9403
    0.05     1.0408     0.8383         0.9396
    0.10     1.0399     0.8377         0.9388
    0.15     1.0392     0.8372         0.9382
    0.20     1.0384     0.8368         0.9376
    0.25     1.0377     0.8365         0.9371
    0.30     1.0370     0.8363         0.9366
    0.35     1.0364     0.8361         0.9362
    0.40     1.0357     0.8359         0.9358
    0.45     1.0352     0.8358         0.9355
    0.50     1.0347     0.8358         0.9352
    0.55     1.0343     0.8357         0.9350
    0.60     1.0339     0.8358         0.9348
    0.65     1.0336     0.8359         0.9347
    0.70     1.0333     0.8361         0.9347 ← best
    0.75     1.0331     0.8363         0.9347
    0.80     1.0330     0.8365         0.9348
    0.85     1.0330     0.8368         0.9349
    0.90     1.0331     0.8372         0.9351
    

## Ensemble Results — Key Findings

**Best blend: 70% GLM + 30% XGBoost (α = 0.70)**

**GLM outperforms pure XGBoost** (MAE 0.9357 vs 0.9403)
The linear model generalises better on unseen data. Football goal
scoring is mostly linear — XGBoost picks up some noise alongside
the real signal.

**Ensemble beats both** (MAE 0.9347)
A small but consistent improvement over either model alone.
In a tight prediction problem every decimal counts.

**The improvement is modest**
Gap between pure GLM and ensemble is only 0.001 — the GLM is
doing most of the heavy lifting. XGBoost adds a small refinement,
particularly at the extremes where non-linear effects are strongest.

**Away goals easier to predict than home**
Away MAE (~0.84) consistently lower than home MAE (~1.03) across
all alpha values. Away scoring has less variance — less influenced
by crowd, travel fatigue, and match context.

In [18]:
# Apply ensemble blend 
# Apply best alpha to generate final blended λ for train and val sets.
# Then apply scoreline matrix and Dixon-Coles to get W/D/L probabilities.

BEST_ALPHA = 0.70

# Blend λ on train set
train['lambda_home_ens'] = BEST_ALPHA * train['lambda_home_glm'] + (1 - BEST_ALPHA) * train['lambda_home_xgb']
train['lambda_away_ens'] = BEST_ALPHA * train['lambda_away_glm'] + (1 - BEST_ALPHA) * train['lambda_away_xgb']

# Blend λ on val set
val['lambda_home_ens'] = BEST_ALPHA * val['lambda_home_glm'] + (1 - BEST_ALPHA) * val['lambda_home_xgb']
val['lambda_away_ens'] = BEST_ALPHA * val['lambda_away_glm'] + (1 - BEST_ALPHA) * val['lambda_away_xgb']

# Apply asymmetric cap — same as Notebook 3
LAMBDA_CAP_HOME = 6.0
LAMBDA_CAP_AWAY = 5.0

train['lambda_home_ens'] = train['lambda_home_ens'].clip(upper=LAMBDA_CAP_HOME)
train['lambda_away_ens'] = train['lambda_away_ens'].clip(upper=LAMBDA_CAP_AWAY)
val['lambda_home_ens']   = val['lambda_home_ens'].clip(upper=LAMBDA_CAP_HOME)
val['lambda_away_ens']   = val['lambda_away_ens'].clip(upper=LAMBDA_CAP_AWAY)

print("Ensemble λ summary — validation set:")
print(f"  Actual home goals:     {val['home_score'].mean():.3f}")
print(f"  GLM    λ_home mean:    {val['lambda_home_glm'].mean():.3f}")
print(f"  XGBoost λ_home mean:   {val['lambda_home_xgb'].mean():.3f}")
print(f"  Ensemble λ_home mean:  {val['lambda_home_ens'].mean():.3f}")

print(f"\n  Actual away goals:     {val['away_score'].mean():.3f}")
print(f"  GLM    λ_away mean:    {val['lambda_away_glm'].mean():.3f}")
print(f"  XGBoost λ_away mean:   {val['lambda_away_xgb'].mean():.3f}")
print(f"  Ensemble λ_away mean:  {val['lambda_away_ens'].mean():.3f}")

Ensemble λ summary — validation set:
  Actual home goals:     1.601
  GLM    λ_home mean:    1.704
  XGBoost λ_home mean:   1.721
  Ensemble λ_home mean:  1.709

  Actual away goals:     1.114
  GLM    λ_away mean:    1.116
  XGBoost λ_away mean:   1.124
  Ensemble λ_away mean:  1.118


## Ranked Probability Score (RPS) — Model Calibration

### What is RPS?
RPS measures how well calibrated the **probability distributions** are —
not just whether we predicted the right winner, but whether we had
the right confidence level.

- **RPS = 0** → perfect prediction
- **RPS = 1** → worst possible prediction
- **Lower is always better**

### Why RPS over accuracy?
A model that says "99% home win" and gets it wrong is penalised much
more than one that says "60% home win" and gets it wrong. Accuracy
treats both the same — RPS doesn't.

This matters for a tournament simulator. We don't just need the right
winner — we need honest probabilities across all outcomes so the
Monte Carlo simulation reflects real uncertainty.

### Formula
For a 3-outcome match (Win / Draw / Loss):  
RPS = 0.5 × [(P(H) - O(H))² + (P(H)+P(D) - O(H)+O(D))²]

Where P = predicted cumulative probability, O = actual outcome.
Cumulative scoring means being wrong about draw vs loss is penalised
less than being wrong about win vs loss.

In [20]:
# Scoreline matrix functions 
# Copied from Notebook 3 — required for RPS calibration and ensemble evaluation
# These convert λ values into full scoreline probability matrices

from scipy.stats import poisson

def scoreline_matrix(lam_home, lam_away, max_goals=8, rho=-0.1):
    home_probs = np.array([poisson.pmf(i, lam_home) for i in range(max_goals + 1)])
    away_probs = np.array([poisson.pmf(j, lam_away) for j in range(max_goals + 1)])
    matrix = np.outer(home_probs, away_probs)

    # Dixon-Coles correction
    matrix[0, 0] *= 1 - lam_home * lam_away * rho
    matrix[1, 0] *= 1 + lam_away * rho
    matrix[0, 1] *= 1 + lam_home * rho
    matrix[1, 1] *= 1 - rho

    matrix /= matrix.sum()
    return matrix


def match_probabilities(lam_home, lam_away, max_goals=8, rho=-0.1):
    mat = scoreline_matrix(lam_home, lam_away, max_goals, rho)

    home_win = float(np.sum(np.tril(mat, -1)))
    draw     = float(np.sum(np.diag(mat)))
    away_win = float(np.sum(np.triu(mat, 1)))

    flat = [(mat[i, j], i, j) for i in range(max_goals + 1)
                               for j in range(max_goals + 1)]
    top5 = sorted(flat, reverse=True)[:5]

    return {
        'home_win':      round(home_win, 4),
        'draw':          round(draw, 4),
        'away_win':      round(away_win, 4),
        'top_scorelines': top5
    }

print("scoreline_matrix and match_probabilities defined")

scoreline_matrix and match_probabilities defined


In [21]:
# Ranked Probability Score (RPS) 
# RPS measures how well calibrated the probability distributions are.
# Unlike accuracy (did we get the winner right?), RPS rewards
# having the RIGHT CONFIDENCE level.
#
# RPS = 0    → perfect prediction
# RPS = 1    → worst possible prediction
# Lower is always better.
#
# For a 3-outcome match (W/D/L):
# RPS = 0.5 × [(P(H) - O(H))² + (P(H)+P(D) - O(H)-O(D))²]
# where P = predicted cumulative probability, O = actual outcome

from scipy.stats import poisson

def compute_rps(home_win_prob, draw_prob, away_win_prob,
                actual_home, actual_away):
    """
    Computes Ranked Probability Score for a single match.
    Actual outcome derived from scoreline.
    """
    # Actual outcome as one-hot
    if actual_home > actual_away:
        o1, o2, o3 = 1, 0, 0   # home win
    elif actual_home == actual_away:
        o1, o2, o3 = 0, 1, 0   # draw
    else:
        o1, o2, o3 = 0, 0, 1   # away win

    # Cumulative predicted probabilities
    p1 = home_win_prob
    p2 = home_win_prob + draw_prob

    # Cumulative actual probabilities
    c1 = o1
    c2 = o1 + o2

    rps = 0.5 * ((p1 - c1)**2 + (p2 - c2)**2)
    return rps


def apply_rps(df, lam_home_col, lam_away_col):
    """
    Apply scoreline matrix and compute RPS for every match in df.
    """
    rps_scores = []
    for _, row in df.iterrows():
        probs = match_probabilities(
            row[lam_home_col],
            row[lam_away_col]
        )
        rps = compute_rps(
            probs['home_win'],
            probs['draw'],
            probs['away_win'],
            row['home_score'],
            row['away_score']
        )
        rps_scores.append(rps)
    return np.mean(rps_scores)


print("Computing RPS on validation set...")

rps_glm = apply_rps(val, 'lambda_home_glm', 'lambda_away_glm')
rps_xgb = apply_rps(val, 'lambda_home_xgb', 'lambda_away_xgb')
rps_ens = apply_rps(val, 'lambda_home_ens', 'lambda_away_ens')

print(f"\nRPS Calibration — Validation Set (lower is better)")
print(f"  GLM:      {rps_glm:.4f}")
print(f"  XGBoost:  {rps_xgb:.4f}")
print(f"  Ensemble: {rps_ens:.4f}")
print(f"\n  Best model: {'Ensemble' if rps_ens < rps_glm and rps_ens < rps_xgb else 'GLM' if rps_glm < rps_xgb else 'XGBoost'}")

Computing RPS on validation set...

RPS Calibration — Validation Set (lower is better)
  GLM:      0.1671
  XGBoost:  0.1683
  Ensemble: 0.1670

  Best model: Ensemble


## RPS Calibration — Findings

**Ensemble wins with RPS 0.1670** — confirmed best model by two
independent metrics (MAE and RPS). Consistent results across both
metrics give confidence the ensemble is genuinely better, not just
lucky on one measure.

**GLM almost matches ensemble** (RPS 0.1671 — gap of 0.0001)
The linear model is remarkably strong for football prediction.
Football is noisy enough that complex models rarely dominate
simple ones by large margins.

**XGBoost trails** (RPS 0.1683 — gap of 0.0013 vs ensemble)
Picked up some noise in training that hurt generalisation.
The 70/30 blend correctly down-weights it.

**0.167 in context**
Bookmaker models typically achieve RPS ~0.155-0.160. We're slightly
behind — expected since bookmakers have access to injury news, team
sheets, and live market information. For a pure statistical model
this is a solid baseline before adding weather and heat features
in the simulator.

In [22]:
# Save everything 
import pickle

models_dir    = Path.home() / 'wc2026/models'
processed_dir = Path.home() / 'wc2026/data/processed'

#  Save XGBoost models as JSON 
# JSON is lighter and more portable than pickle for XGBoost native models

xgb_home.save_model(str(models_dir / 'xgb_home.json'))
xgb_away.save_model(str(models_dir / 'xgb_away.json'))

# Save ensemble alpha 
with open(models_dir / 'ensemble_alpha.pkl', 'wb') as f:
    pickle.dump(BEST_ALPHA, f)

# Save utility functions 
# Save scoreline_matrix and match_probabilities so every notebook
# can import them without redefining
import inspect

utils_code = inspect.getsource(scoreline_matrix) + "\n\n" + \
             inspect.getsource(match_probabilities)

with open(models_dir / 'utils.py', 'w') as f:
    f.write("import numpy as np\n")
    f.write("from scipy.stats import poisson\n\n")
    f.write(utils_code)

# Save final training predictions 
df_final = pd.concat([train, val]).sort_values('date').reset_index(drop=True)

df_final.to_csv(processed_dir / 'training_predictions_ensemble.csv', index=False)

# Summary 
print("Saved:")
print(f"  {models_dir}/xgb_home.json")
print(f"  {models_dir}/xgb_away.json")
print(f"  {models_dir}/ensemble_alpha.pkl")
print(f"  {models_dir}/utils.py")
print(f"  {processed_dir}/training_predictions_ensemble.csv")

print(f"\nModel performance summary:")
print(f"  Best alpha:       {BEST_ALPHA}")
print(f"  RPS GLM:          {rps_glm:.4f}")
print(f"  RPS XGBoost:      {rps_xgb:.4f}")
print(f"  RPS Ensemble:     {rps_ens:.4f}")
print(f"  Training rows:    {len(df_final):,}")


Saved:
  /home/ubuntu/wc2026/models/xgb_home.json
  /home/ubuntu/wc2026/models/xgb_away.json
  /home/ubuntu/wc2026/models/ensemble_alpha.pkl
  /home/ubuntu/wc2026/models/utils.py
  /home/ubuntu/wc2026/data/processed/training_predictions_ensemble.csv

Model performance summary:
  Best alpha:       0.7
  RPS GLM:          0.1671
  RPS XGBoost:      0.1683
  RPS Ensemble:     0.1670
  Training rows:    24,714
